### Setup

In [1]:
from tenk_rag.embed import load_chunks
from tenk_rag.evaluate import load_questions
from tenk_rag.search import Searcher
from tenk_rag.bm25 import BM25Searcher
from tenk_rag.rewrite import Rewriter

questions = {q["id"]: q for q in load_questions()}
dense = Searcher()
bm25 = BM25Searcher(dense.chunks)
rewriter = Rewriter()

# If you get 'KeyError: 'id'', check theid field name in questions.jsonl and change it.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Inspect Gold Chunks

In [2]:
def has_evidence(chunk, evidence):
    """Checks whether a chunk contains all evidence strings, the same rule the evaluation uses."""
    text = chunk["text"].lower()
    return all(e.lower() in text for e in evidence)

def show_gold(qid, max_chars=1500):
    """Prints the question, its rewrite, and the evidence, then finds and prints every gold chunk for that company."""
    q = questions[qid]
    print(qid, "|", q["question"])
    print("rewrite :", rewriter.rewrite(q["question"]))
    print("evidence:", q["evidence"])
    gold = [c for c in dense.chunks if c["ticker"] == q["ticker"] and has_evidence(c, q["evidence"])]
    print("gold chunks:", len(gold))
    for c in gold:
        print("-" * 80)
        print(c["id"])
        print(c["text"][:max_chars])

show_gold("xom-02")

xom-02 | How much did ExxonMobil spend on research and development in 2025?
rewrite : How much did ExxonMobil spend on research and development expense in 2025?
evidence: ['Research and development', '1,228']
gold chunks: 1
--------------------------------------------------------------------------------
XOM-46-1
Table of Contents Financial Table of Contents FINANCIAL INFORMATION
| (millions of dollars, except where stated otherwise) | 2025 | 2024 | 2023 |
| --- | --- | --- | --- |
| Additions to property, plant, and equipment (1) | 31,476 | 109,332 | 29,038 |
| Property, plant, and equipment, less allowances | 299,373 | 294,318 | 214,940 |
| Total assets | 448,980 | 453,475 | 376,317 |
| Exploration expenses, including dry holes | 1,007 | 826 | 751 |
| Research and development costs | 1,228 | 987 | 879 |
| Long-term debt | 34,241 | 36,755 | 37,483 |
| Total debt | 43,537 | 41,710 | 41,573 |
| Debt to capital (percent) | 14.0 | 13.4 | 16.4 |
| Net debt to capital (percent) (2) | 11.0 | 

In [3]:
show_gold("msft-03")

msft-03 | How much did Microsoft invest in developing new technology and products in fiscal 2024?
rewrite : What was Microsoft's research and development expense in fiscal 2024?
evidence: ['Research and development', '29,510']
gold chunks: 1
--------------------------------------------------------------------------------
MSFT-47-0
ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA INCOME STATEMENTS
| (In millions, except per share amounts) Year Ended June 30, | 2026 | 2025 | 2024 |
| --- | --- | --- | --- |
| Revenue - Product | 64,696 | 63,946 | 64,773 |
| Revenue - Service and other | 267,143 | 217,778 | 180,349 |
| Total revenue | 331,839 | 281,724 | 245,122 |
| Cost of revenue - Product | 12,098 | 13,501 | 15,272 |
| Cost of revenue - Service and other | 94,276 | 74,330 | 58,842 |
| Total cost of revenue | 106,374 | 87,831 | 74,114 |
| Gross margin | 225,465 | 193,893 | 171,008 |
| Research and development | 35,562 | 32,488 | 29,510 |
| Sales and marketing | 26,710 | 25,654 | 24,4

### Find the Gold Chunk's Full Rank

In [4]:
def gold_rank(results, evidence):
    for rank, (chunk, _) in enumerate(results, start=1):
        if has_evidence(chunk, evidence):
            return rank
    return None

def diagnose(qid):
    q = questions[qid]
    for label, query in [("original", q["question"]), ("rewrite", rewriter.rewrite(q["question"]))]:
        d = gold_rank(dense.search(query, q["ticker"], k=100000), q["evidence"])
        b = gold_rank(bm25.search(query, q["ticker"], k=100000), q["evidence"])
        print(f"{qid} {label:<9} dense rank: {d}   bm25 rank: {b}")

diagnose("xom-02")
diagnose("msft-03")

xom-02 original  dense rank: 198   bm25 rank: 42
xom-02 rewrite   dense rank: 152   bm25 rank: 58
msft-03 original  dense rank: 143   bm25 rank: 287
msft-03 rewrite   dense rank: 112   bm25 rank: 45


### Insepct the Chunks That Outranked It

In [5]:
"""Shows the chunks that took ranks 1-3 instead of the answer for the rewritten query.

This tests hypothesis 2 (too many competing chunks)."""
def show_top(qid, n=3, max_chars=500):
    q = questions[qid]
    query = rewriter.rewrite(q["question"])
    for name, searcher in [("dense", dense), ("bm25", bm25)]:
        print("=" * 80, name)
        for chunk, score in searcher.search(query, q["ticker"], k=n):
            print(f"{chunk['id']}  score={score:.3f}")
            print(chunk["text"][:max_chars])
            print("=" * 40)

show_top("xom-02")

================================================================================ dense
XOM-98-0  score=0.700
Throughout ExxonMobil’s businesses, new and ongoing measures are taken to prevent and minimize the impact of our operations on air, water, and ground. These include significant investments in refining infrastructure and technology to manufacture clean fuels; projects to monitor and reduce air, water, and waste emissions, both from the Company’s operations and from other companies; and expenditures for asset retirement obligations. Using definitions and guidelines established by the American Petr
XOM-65-10  score=0.626
such, the Outlook does not project the degree of required future policy and technology advancement and deployment for the world, or ExxonMobil, to meet net zero by 2050. As future policies and technology advancements emerge, they will be incorporated into the Outlook, and ExxonMobil’s business plans will be updated accordingly. References to projects or opportuniti

In [6]:
show_top("msft-03")

================================================================================ dense
MSFT-30-0  score=0.704
Research and development expenses include payroll, stock-based compensation expense, employee benefits, and other headcount-related expenses associated with product development. Research and development expenses also include technology development costs, including AI training and other infrastructure costs, third-party development and programming costs, and the depreciation and amortization of assets used to conduct research and development. Fiscal Year 2026 Compared with Fiscal Year 2025 Resear
MSFT-88-0  score=0.639
Following are the supplemental consolidated financial results of Microsoft Corporation on an unaudited pro forma basis, as if the acquisition had been consummated on July 1, 2022:
MSFT-56-17  score=0.616
programs as we have determined annual compensation is commensurate with annual sales activities. Cost of Revenue Cost of revenue includes: costs incurred to suppo